In [ ]:
import matplotlib       
import matplotlib.pyplot as plt
import numpy as np      
import pandas as pd     
import rushd as rd      
import scipy as sp      
import seaborn as sns   

# Set seaborn style
sns.set_style('ticks')
sns.set_context('talk', rc={'font.family': 'sans-serif', 'font.sans-serif':['Helvetica Neue']})

While ddPCR can be used for several different kinds of measurements, the most common one we use in lab is copy number variation (CNV) analysis, a.k.a. determining the copy number of a gene in a (presumably) homogeneous sample.

# Example: Copy number calculation

In this experiment, we measured the copy number of the TANGLES PiggyBac-integrated, monoclonal HEK293T cell lines. We extracted gDNA from five cell lines and additionally from three of those lines that had been sorted on highly expressing cells. (Since these are monoclones, the copy numbers should be the same; however, with HEK293T cells some genetic drift may occur.) For each of these samples, we performed two ddPCR assays to detect copy number of the *mRuby2* and *PuroR* genes separately (FAM channel). These genes were integrated at the same time but on separate vectors, so their copy numbers likely differ within the same cell line. For both assays, we also included assays (probes + primer set) to simultaneously detect the reference gene *RPP30* (common reference for human cells) in the HEX channel. This is essential for computing copy number rather than simply the concentration of the input material. Formulas for this calculation are shown below.

## Load data

Use the `rushd` function to load straight from the `.ddpcr` file outputted from the BioRad machine. Since we've specified all the relevant metadata in the `.yaml` file, we don't need to load the metadata from the instrument. The metadata fields added are:

- `cell_line`: the cell line ID (`cTA#`) for which copy number is being measured
- `sorted`: whether the cell line was sorted on highly expressing cells prior to measurement
- `FAM_target`: the gene measured by the FAM probe; here, either the *mRuby2* or *PuroR* gene
- `HEX_target`: the gene measured by the HEX probe; here and usually, this is the reference gene *RPP30*

Note that if metadata is extracted from the experiment file, it will include the fields `FAM_target`, `HEX_target`, and `condition#` (where `#` is 1-4, corresponding to "Sample Description #" fields in the software).

The actual data loaded are the `HEX` and `FAM` columns, which display the fluorescence in each channel for each droplet, just like for cells in flow cytometry. Each row of the DataFrame represents a droplet.

There's no need to cache the data since the file is so small, but feel free to do so if you prefer.

In [ ]:
base_path = rd.datadir/'data'/'Weiss.lab.ddPCR'/'2025.07.18_ddPCR.TANGLES.monoclones_KL'
data_path = base_path/'2025.07.18_ddPCR.TANGLES_DSP-KL.ddpcr'
yaml_path = base_path/'wells.yaml'

output_path = rd.rootdir/'output'/'ddPCR-example'
cache_path = output_path/'data.gzip'

rd.plot.plot_well_metadata(yaml_path)

In [ ]:
df = rd.ddpcr.load_ddpcr(data_path, yaml_path, extract_metadata=False)
df.dropna(subset='cell_line', inplace=True)
display(df)

In [ ]:
# To make plotting easier
df['condition'] = df['cell_line'] + df['sorted'].map({False: '', True: '_sorted'})

## Plot droplets

In [ ]:
# Manually draw gates for each target
# since there is no negative control condition
gates = {
    'mRuby2': 1e3,
    'PuroR': 2e3,
    'RPP30': 1.5e3,
}
g = sns.relplot(data=df, x='HEX', y='FAM', row='FAM_target', col='condition',
                facet_kws=dict(margin_titles=True))
for (fam,condition), ax in g.axes_dict.items():
    ax.axvline(gates['RPP30'], color='black', zorder=0)
    ax.axhline(gates[fam], color='black', zorder=0)

## Count droplets and compute copy number

In [ ]:
# Compute fraction positive for each channel
by = ['cell_line', 'sorted', 'condition', 'FAM_target', 'HEX_target']
counts = df.groupby(by)[df.columns[0]].count().rename('total').reset_index()
hex_negative = df.groupby(by)[['HEX_target','HEX']].apply(lambda x: np.sum(x['HEX'] < gates[x['HEX_target'].values[0]])).rename('HEX_negative').reset_index()
fam_negative = df.groupby(by)[['FAM_target','FAM']].apply(lambda x: np.sum(x['FAM'] < gates[x['FAM_target'].values[0]])).rename('FAM_negative').reset_index()
counts = counts.merge(hex_negative, on=by, how='left').merge(fam_negative, on=by, how='left')
display(counts)


Ideally, there should be 10,000 droplets per condition, and at least 1,000 negative droplets to properly draw the gates. This experiment does not have the latter: we loaded too much DNA input, so almost all the droplets are FAM-positive in some of the conditions. We'll estimate gates from the other conditions, but the quantification would be better with more negatives.

In [ ]:
# Display conditions with too few droplets
totals_low = counts[counts['total']<1e4]
if totals_low.empty: print('All conditions have >10,000 droplets!')
else: display(totals_low)

for channel in ['HEX', 'FAM']:
    channel_low = counts[counts[f'{channel}_negative']<1e3]
    if channel_low.empty: print(f'All conditions have >1,000 {channel}-negative droplets!')
    else: display(channel_low)

Now we use the Poisson distribution to compute copies per droplet. This [resource](https://www.bio-rad.com/webroot/web/pdf/lsr/literature/Bulletin_6407.pdf) from BioRad describes the formula for computing copy number, as well as other helpful tips. For convenience, the derivation is repeated here.

Assuming a Poisson process for partitioning amplified sequences into droplets, the probability that a droplet will contain $n$ copies given a mean number of copies per droplet $C$ is:

$\begin{equation} P(n) = \frac{C^n e^{-C}}{n!} \end{equation}$

Since we can't easily translate fluorescence into a copy number for a single droplet, we can instead consider droplets *lacking* a copy of the sequence. The probability that a droplet contains zero copies is the following:

$
\begin{align}
P(n=0) &= \frac{C^0 e^{-C}}{0!} \\
&= e^{-C}
\end{align}
$

This gives the probability that any single droplet contains zero copies; across an entire population, the fraction of droplets containing zero copies is a good estimate of this value. Thus, for a total number of droplets $D_t$ and number of empty droplets $D_0$, we have:

$
\begin{align}
P(n=0) &\approx \frac{D_0}{D_t} \\
e^{-C} &= \frac{D_0}{D_t} \\
C &= -\ln{\frac{D_0}{D_t}} \\
C &= \ln{\frac{D_t}{D_0}}
\end{align}
$

Now, we can compare the number of copies per droplet of the target gene ($C_{target}$) to that of the reference gene ($C_{ref}$) obtain a copy number ($N_{target}$) in our cell line. This requires that we know the copy number of the reference gene ($N_{ref}$), as we will be comparing ratios of copy numbers per droplet to copy numbers in the genome.

$
\begin{align}
\frac{N_{target}}{N_{ref}} &= \frac{C_{target}}{C_{ref}}\\
N_{target} &= N_{ref} \frac{C_{target}}{C_{ref}}
\end{align}
$

Now we can combine this with the equation for copies per droplet to find the genome copy number of our target.

For HEK293T cells, it was empirically determined that the reference gene *RPP30* has a copy number of ~2.7, a fractional number since the genome of these cells is unstable. For hiPSCs, the copy number of *RPP30* is the expected 2. Other good references are known single (or double) integrations of particular a cargo at STRAIGHT-IN landing pads.

In [ ]:
# Compute copies per droplet
counts['HEX_copies_per_droplet'] = np.log2(counts['total'] / counts['HEX_negative'])
counts['FAM_copies_per_droplet'] = np.log2(counts['total'] / counts['FAM_negative'])
counts['HEX_copy_num'] = 2.7    # empirically determined RPP30 copy number in HEK293T cells
counts['FAM_copy_num'] = counts['HEX_copy_num'] * counts['FAM_copies_per_droplet'] / counts['HEX_copies_per_droplet']
display(counts)

In [ ]:
plot_df = counts.sort_values(['cell_line','sorted'])
f = sns.scatterplot(data=plot_df, x='condition', y='FAM_copy_num', hue='FAM_target')
_ = f.set_xticklabels(f.get_xticklabels(), rotation=90)
f.set(ylabel='Copy number', xlabel='')

From this data, we notice that the copy number of the *PuroR* gene is higher than that of the *mRuby2* gene in all the cell lines. This makes sense, as the vector containing mRuby2 was larger (two genes), and selective pressure was only applied to the PuroR cassette. We also can see that the sorted and unsorted values for the three tested lines differ, possibly reflecting the genome instability we expect in HEK293T cells.

## Calculate confidence interval on copy number

But what are the error bars on this value? Rather than running the experiment again, we want to estimate our confidence in the calculated values. To do so, we can use bootstrapping to resample the droplet DataFrame and re-compute copy number on each of these samples. The resulting distribution of computed copy numbers represents a reasonable range in which the actual copy number lies.

In [ ]:
def calculate_copy_number(fam_negative, hex_negative):
    ''' 
    This function takes the equivalent of the `counts['FAM_negative']`
    and `counts['HEX_negative']` columns from above.
    '''
    copies_fam = np.log(len(fam_negative) / fam_negative.sum())
    copies_hex = np.log(len(hex_negative) / hex_negative.sum())

    # Calculate copy number relative to reference
    hex_copy_num = 2.7 # for RPP30 in HEK293T
    fam_copy_num = hex_copy_num * copies_fam / copies_hex

    return fam_copy_num

def perform_bootstrapping(x):
    ''' 
    This function performs bootstrapping on a single group of a DataFrame.
    The scipy `bootstrap` function requires data to be passed as numpy arrays,
    so we'll extract the relevant columns of the DataFrame and pass them along.

    Notice that what is being resampled here are the `FAM_negative` and 
    `HEX_negative` columns directly (i.e., True/False values). This is equivalent
    to sampling the raw channel values and gating them, since the gates don't 
    change based on the sample.
    '''
    x_data = (x['FAM_negative'].to_numpy(), x['HEX_negative'].to_numpy())
    result = sp.stats.bootstrap(x_data, calculate_copy_number, random_state=2026, method='basic')
    return result.confidence_interval

df['HEX_negative'] = df['HEX'] < df['HEX_target'].map(gates)
df['FAM_negative'] = df['FAM'] < df['FAM_target'].map(gates)

ci = df.groupby(by).apply(perform_bootstrapping).rename('ci').reset_index()
counts = counts.merge(ci, on=by, how='left')
display(counts)

Unfortunately, confidence intervals in the form of tuples are not particularly useful for plotting. The `matplotlib` errorbar function 
takes the argument `yerr` (or `xerr`) of shape (2,N), where the first row contains the lower error and the second row contains the upper error, and the columns correspond to data points. Here, the error is the offset from the plotted value, not the location of the error bar endpoint. So we'll transform the `ci` data into this form for plotting.

In [ ]:
def compute_error_amounts(df):
    '''
    Compute the matplotlib desired errors.
    First, transform the confidence interval tuple into separate 
    columns. Then, convert this array of arrays to a proper 2D array, 
    and transpose it to match the matplotlib expected shape (2,N).
    Finally, convert these interval bounds into offsets.
    '''
    error_vals = np.stack(df['ci'].apply(lambda x: np.array([x.low, x.high])).to_numpy()).transpose()
    center = df['FAM_copy_num'].to_numpy()
    
    return np.abs(error_vals - center)

In [ ]:
# Plot copy number with errorbars (confidence interval)
plot_df = counts.sort_values(['cell_line','sorted'])
f = sns.scatterplot(data=plot_df, x='condition', y='FAM_copy_num', hue='FAM_target')
_ = f.set_xticklabels(f.get_xticklabels(), rotation=90)
f.set(ylabel='Copy number', xlabel='')

for fam_target, this_df in plot_df.groupby('FAM_target'):
    f.errorbar(x=this_df['condition'], y=this_df['FAM_copy_num'], yerr=compute_error_amounts(this_df),
               linestyle='none', capsize=4, capthick=2, color='black',)

Notice that the errorbars a quite large for the high copy numbers. This makes sense because these conditions had a very small number of negative droplets, so resampling will introduce greater variability in the fraction negative used in the copy number calculation.

Hopefully soon, we will add this copy number calculation to `rushd` for easier analysis :)